# Airfoil surrogate v2 - corrected geometry, log drag, honest evaluation

Run training and accuracy experiments on Kaggle via the API with **GPU enabled**.
The default GPU setup fails before training if CUDA is unavailable. An explicit
API CPU override runs the same accuracy candidates on Kaggle CPU, never a silent fallback.
Before running:

1. **Add Input** -> Datasets -> add the pushed tables dataset (`airfoil_geometry.csv`,
   `airfoil_polars.csv`, `load_report.json`). Create it locally with
   `python -m src.utils.kaggle_sync push <owner>/<slug> --create`.
2. Enable GPU and Internet in the API kernel settings. The API injects a hashed
   source bundle; browser runs without that bundle fall back to a Git checkout.
3. Push the staged notebook via the API, then pull its saved results with
   `python -m src.utils.kaggle_sync pull <owner>/<kernel-slug>`.

What changed from v1, and why the numbers are not directly comparable:

- v1 normalized each surface independently and subtracted its own mean `y`, which
  destroyed thickness and camber: 43% of sampled stations had the upper surface
  *below* the lower one. This changes the physical representation, but does not
  establish that deleting thickness information caused memorization. Held-out
  geometry and domain-shift diagnostics are needed to explain generalization.
- v2 normalizes both surfaces with one shared transform, samples a cosine-spaced
  grid, and stores a physical thickness/camber basis plus named descriptors.
- `Cd` is fitted as `log(Cd)` and reported in physical units, with relative error
  alongside absolute, because v1's `Cd` MAE of 0.0049 is roughly 80% relative error
  on a low-drag cruise section.
- The geometry split is reused verbatim from the v1 baseline, so the held-out
  airfoils are identical; no rows or extreme geometries are removed.
- The completed 20261013 v2 baseline MLP stopped at 58/100 iterations with
  tolerance 0.0001 and no convergence warning, not at the iteration cap. Its
  test `Cd` R-squared was 0.6393 and median relative error was 8.11%. The held-out
  `fx79w660a` contributed 80.9% of `Cd` squared error; its thickness 0.664 exceeds
  the training maximum 0.478. These are observed baseline facts, not new results.

In [ ]:
from pathlib import Path
import hashlib
import subprocess
import sys

if not Path('/kaggle/input').is_dir() or not Path('/kaggle/working').is_dir():
    raise RuntimeError('Run this notebook on Kaggle with GPU enabled, not locally.')
ACCELERATOR = globals().get('KAGGLE_ACCELERATOR', 'gpu')
if ACCELERATOR not in ('gpu', 'cpu'):
    raise ValueError('KAGGLE_ACCELERATOR must be gpu or an explicit cpu override.')

REPO_URL = 'https://github.com/gaurav-zz/ML-for-CFD.git'
REPO_BRANCH = 'v2-physical-geometry'

# Clone into scratch, not /kaggle/working: everything under working becomes the
# kernel's output, and the checkout is not a result worth publishing.
SCRATCH = Path('/kaggle/temp') if Path('/kaggle/temp').is_dir() else Path('/tmp')
if 'BUNDLED_SOURCE_DIR' in globals():
    if not all(name in globals() for name in ('BUNDLED_SOURCE_SHA256', 'BUNDLED_GIT_REVISION')):
        raise RuntimeError('Injected source bundle is missing its hash or Git revision.')
    PROJECT_ROOT = Path(BUNDLED_SOURCE_DIR).resolve()
    SOURCE_REVISION = BUNDLED_GIT_REVISION
    SOURCE_SHA256 = BUNDLED_SOURCE_SHA256
    SOURCE_MODE = 'bundle'
    if not isinstance(SOURCE_SHA256, str) or len(SOURCE_SHA256) != 64 or any(
        character not in '0123456789abcdef' for character in SOURCE_SHA256
    ):
        raise ValueError('Injected source bundle requires a SHA256 hexadecimal digest.')
    if not (PROJECT_ROOT / 'src').is_dir():
        raise FileNotFoundError(f'Bundled source not found at {PROJECT_ROOT}.')
else:
    PROJECT_ROOT = SCRATCH / 'airfoil-ml-surrogate'
    if not PROJECT_ROOT.exists():
        subprocess.run(
            ['git', 'clone', '--branch', REPO_BRANCH, REPO_URL, str(PROJECT_ROOT)], check=True,
        )
    if not (PROJECT_ROOT / '.git').exists():
        raise FileNotFoundError(f'Expected a Git checkout at {PROJECT_ROOT}. Use a fresh session.')
    if subprocess.check_output(['git', 'status', '--porcelain'], cwd=PROJECT_ROOT, text=True).strip():
        raise RuntimeError('The source checkout has local edits. Use a fresh Kaggle session.')
    previous_revision = subprocess.check_output(
        ['git', 'rev-parse', 'HEAD'], cwd=PROJECT_ROOT, text=True,
    ).strip()
    subprocess.run(['git', 'pull', '--ff-only', 'origin', REPO_BRANCH], cwd=PROJECT_ROOT, check=True)
    SOURCE_REVISION = subprocess.check_output(
        ['git', 'rev-parse', 'HEAD'], cwd=PROJECT_ROOT, text=True,
    ).strip()
    if SOURCE_REVISION != previous_revision and any(
        name == 'src' or name.startswith('src.') for name in sys.modules
    ):
        raise RuntimeError('Source updated after project modules were imported. Restart and run all.')
    SOURCE_SHA256 = hashlib.sha256(subprocess.check_output(
        ['git', 'archive', SOURCE_REVISION], cwd=PROJECT_ROOT,
    )).hexdigest()
    SOURCE_MODE = 'git-archive'

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     '-r', str(PROJECT_ROOT / 'requirements.txt'),
     '-r', str(PROJECT_ROOT / 'requirements-kaggle.txt')],
    check=True,
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
import torch

if ACCELERATOR == 'gpu' and not torch.cuda.is_available():
    raise RuntimeError('CUDA is unavailable. Enable a Kaggle GPU accelerator and rerun.')
GPU_METADATA = {
    'torch_version': str(torch.__version__),
    'cuda_version': torch.version.cuda,
    'device_name': torch.cuda.get_device_name(torch.cuda.current_device()) if ACCELERATOR == 'gpu' else None,
    'device_count': torch.cuda.device_count() if ACCELERATOR == 'gpu' else 0,
}
print(f'Source: {PROJECT_ROOT} at {SOURCE_REVISION}; {SOURCE_MODE} SHA256={SOURCE_SHA256}')
print(f'Requested accelerator: {ACCELERATOR}; actual GPU runtime:', GPU_METADATA)

In [ ]:
import os

GPU_TESTS_PASSED = CPU_TESTS_PASSED = False
if not Path('/kaggle/input').is_dir():
    raise RuntimeError('Fitting tests must run on Kaggle, not locally.')
if ACCELERATOR == 'gpu' and not torch.cuda.is_available():
    raise RuntimeError('GPU fitting tests must run on Kaggle with CUDA available.')
subprocess.run(
    [sys.executable, '-m', 'pytest', '-q', 'tests',
     '-k', 'not test_notebook_runs_the_whole_v2_experiment_on_synthetic_tables'],
    cwd=PROJECT_ROOT, env={**os.environ,
        'KAGGLE_GPU_TESTS': '1' if ACCELERATOR == 'gpu' else '0',
        'KAGGLE_CPU_TESTS': '1' if ACCELERATOR == 'cpu' else '0'}, check=True,
)
GPU_TESTS_PASSED = ACCELERATOR == 'gpu'
CPU_TESTS_PASSED = ACCELERATOR == 'cpu'

## Locate the input tables

The two tables are joined on `geometry_id` in memory, so the attached dataset stays
about 57 MB instead of v1's 1.7 GB CSV. Set `DATASET_DIR` explicitly if more than one
attached dataset contains a geometry table.

In [ ]:
# Leave DATASET_DIR as None to find the attached tables dataset automatically, or
# pin it to the directory holding the CSVs when several datasets carry them.
DATASET_DIR = None
INPUT_ROOT = Path('/kaggle/input')
OUTPUT_ROOT = Path('/kaggle/working')

In [ ]:
from src.data.loaders import GEOMETRY_TABLE, POLARS_TABLE

if DATASET_DIR is None:
    # Search at any depth: Kaggle has mounted attached datasets both flat at
    # /kaggle/input/<slug>/ and nested under /kaggle/input/datasets/, so the
    # layout is not something to assume.
    found = sorted({match.parent for match in INPUT_ROOT.rglob(GEOMETRY_TABLE)})
    if not found:
        present = sorted(p.relative_to(INPUT_ROOT).as_posix()
                         for p in INPUT_ROOT.rglob('*') if p.is_file())[:20]
        raise FileNotFoundError(
            f'No attached dataset contains {GEOMETRY_TABLE}. {INPUT_ROOT} holds: '
            f'{present or "nothing"}. Attach the dataset pushed by src.utils.kaggle_sync, '
            'and check it reports "ready" first: Kaggle mounts nothing for a dataset '
            'whose version is still processing.'
        )
    if len(found) > 1:
        raise RuntimeError(
            f'Several datasets contain {GEOMETRY_TABLE}; set DATASET_DIR to one of {found}.'
        )
    DATASET_DIR = found[0]
DATASET_DIR = Path(DATASET_DIR)

GEOMETRY_PATH = DATASET_DIR / GEOMETRY_TABLE
POLARS_PATH = DATASET_DIR / POLARS_TABLE
FROZEN_SPLIT = PROJECT_ROOT / 'configs' / 'frozen_split.json'
for path in (GEOMETRY_PATH, POLARS_PATH, FROZEN_SPLIT):
    if not path.is_file():
        raise FileNotFoundError(f'Required input not found: {path}')
print(f'Tables: {DATASET_DIR}')
print(f'Frozen split: {FROZEN_SPLIT}')

## Restore the frozen geometry split

`load_frozen_split` reuses the exact `geometry_id` lists the v1 baseline froze, records a
*new* fingerprint for the v2 tables, and re-asserts that the partitions are disjoint and
cover every airfoil. A mismatch stops the run before any training.

In [ ]:
from copy import deepcopy
from datetime import datetime, timezone
from importlib.metadata import version
import gc
import json

import joblib
import numpy as np
import pandas as pd

from src.data.features import profile_columns
from src.training.comparison import (
    ACCURACY_MODEL_CONFIGS, compare_models, evaluate_selected_models, load_frozen_split,
)
from src.training.targets import DEFAULT_TARGET_TRANSFORMS, PhysicalPredictor

TARGET_COLUMNS = ('Cl', 'Cd', 'Cm')

split, feature_columns, target_columns, dataset_metadata = load_frozen_split(
    GEOMETRY_PATH, POLARS_PATH, FROZEN_SPLIT, target_columns=TARGET_COLUMNS,
)
if globals().get('KAGGLE_STUDY_MODE', 'accuracy') == 'auxiliary':
    from src.data.auxiliary import load_auxiliary
    auxiliary_report_path = DATA_DIR / 'auxiliary_report.json'
    auxiliary = load_auxiliary(DATA_DIR / 'airfoil_auxiliary.csv', POLARS_PATH, auxiliary_report_path)
    keys = ['geometry_id', 'Alpha', 'Re', 'Ncrit']
    lookup = auxiliary.set_index(keys)
    for partition in ('train', 'validation', 'test'):
        frame = getattr(split, partition)
        labels = lookup.reindex(pd.MultiIndex.from_frame(frame[keys]))
        if labels['Cdp_observed'].isna().any():
            raise ValueError('Auxiliary sidecar does not cover the frozen row identities.')
        frame['Cdp'] = labels['Cdp'].to_numpy()
        frame['Cdp_observed'] = labels['Cdp_observed'].to_numpy(dtype=bool)
    dataset_metadata['auxiliary'] = json.loads(auxiliary_report_path.read_text())
    del auxiliary, lookup, labels
    print('Cdp attached as a label only; production holdouts remain unscored.')
else:
    X_train, y_train = split.train[feature_columns], split.train[target_columns]
    X_validation, y_validation = split.validation[feature_columns], split.validation[target_columns]

print(f'Features: {len(feature_columns)}; targets: {target_columns}')
for partition in ('train', 'validation', 'test'):
    frame = getattr(split, partition)
    print(f'{partition}: {len(frame):,} rows, {frame.geometry_id.nunique():,} geometries')
print(f"v1 row counts for reference: {dataset_metadata['baseline_split_rows']}")

# The v1 representation left 43% of stations with the upper surface below the
# lower one; this must now read zero.
geometry_table = pd.read_csv(GEOMETRY_PATH, dtype={'geometry_id': str})
stations = geometry_table[[name for name in profile_columns(geometry_table)
                           if name.startswith('thickness_')]].to_numpy()
print(f'Negative thickness stations: {int((stations < 0).sum())} of {stations.size:,}')
print(f"Airfoils whose source coordinates were clipped: "
      f"{int((geometry_table['clipped_stations'] > 0).sum())} of {len(geometry_table):,}")

## Train the queue

The completed benchmark identified MLP as the best family, so the accuracy queue
retains the original `mlp` baseline with 113 features and compares `torch_mlp_mse`
with `torch_mlp_huber` on GPU rather than rerunning the full CPU forest queue.
Both new models internally engineer two-angle features and use reflection symmetry,
SiLU and RAdam. Their losses use a train-only normalized fitting target; Huber
delta is in those normalized units. `scale_targets=False` avoids a second scaler.
The outer geometry-disjoint validation split is passed to fitting after the target
transform, for scheduling and stopping; test data never enter training.
`Cd` is fitted as `log(Cd)`; every metric is inverted back to physical coefficient units
before it is written, so `comparison.csv` stays directly readable.

In [ ]:
MODEL_CONFIGS = deepcopy(ACCURACY_MODEL_CONFIGS)
if ACCELERATOR == 'cpu':
    for config in MODEL_CONFIGS:
        if config['name'].startswith('torch_mlp_'):
            config['parameters']['device'] = 'cpu'
TARGET_TRANSFORMS = dict(DEFAULT_TARGET_TRANSFORMS)
SELECTION_METRICS = {'Cd': 'median_relative_error'}
display(pd.DataFrame(MODEL_CONFIGS))
print(f'Target transforms: {TARGET_TRANSFORMS}')
print('Validation selection: Cl/Cm physical RMSE; Cd median relative error.')

In [ ]:
if ACCELERATOR == 'gpu' and (not GPU_TESTS_PASSED or not torch.cuda.is_available()):
    raise RuntimeError('Run the Kaggle GPU setup and fitting tests before the experiment.')
if ACCELERATOR == 'cpu' and not CPU_TESTS_PASSED:
    raise RuntimeError('Run the Kaggle CPU fitting tests before the experiment.')
run_started = datetime.now(timezone.utc)
RUN_OUTPUT = OUTPUT_ROOT / 'experiments' / 'v2' / run_started.strftime('%Y%m%dT%H%M%S%fZ')
run_metadata = {
    'created_at_utc': run_started.isoformat(),
    'git_commit': SOURCE_REVISION,
    'source_sha256': SOURCE_SHA256,
    'source_mode': SOURCE_MODE,
    'gpu': GPU_METADATA,
    'accelerator': ACCELERATOR,
    'machine_shape': globals().get('KAGGLE_MACHINE_SHAPE'),
    'machine_shape_source': 'requested API setting; actual hardware is gpu.device_name',
    'device': 'cuda' if ACCELERATOR == 'gpu' else 'cpu',
    'inference_device': 'cpu-numpy',
    'study_mode': globals().get('KAGGLE_STUDY_MODE', 'accuracy'),
    'gpu_tests_passed': GPU_TESTS_PASSED,
    'cpu_tests_passed': CPU_TESTS_PASSED,
    'python_version': sys.version,
    'packages': {
        name: version(name)
        for name in ['numpy', 'pandas', 'scikit-learn', 'joblib', 'matplotlib', 'torch', 'pytest']
    },
    'platform': 'kaggle',
    'representation': 'v2-thickness-camber-plus-descriptors',
    **dataset_metadata,
}
validation_results = compare_models(
    x_train=X_train, y_train=y_train,
    x_validation=X_validation, y_validation=y_validation,
    target_names=target_columns, output_dir=RUN_OUTPUT,
    model_configs=MODEL_CONFIGS, run_metadata=run_metadata,
    target_transforms=TARGET_TRANSFORMS,
    selection_metrics=SELECTION_METRICS,
)
print(f'Comparison artifacts: {RUN_OUTPUT}')

## Compare validation performance

Select `Cl` and `Cm` by physical validation RMSE, but `Cd` by validation median
relative error. The winners may differ. The full metric table remains visible;
the plots below show physical RMSE for comparison, not the drag selection rule.
Each model's saved result includes its training metadata and stopping diagnostics.

In [ ]:
import matplotlib.pyplot as plt

comparison_report = json.loads((RUN_OUTPUT / 'run.json').read_text())
failed_models = [
    name for name, result in comparison_report['results'].items() if result['status'] == 'failed'
]
if failed_models:
    print(f'Models that failed: {failed_models}. See run.json for diagnostics.')
if validation_results.empty:
    raise RuntimeError(f"No models completed successfully. Inspect {RUN_OUTPUT / 'run.json'}.")
for target in target_columns:
    selection_metric = SELECTION_METRICS.get(target, 'rmse')
    scores = validation_results[validation_results.target == target]
    display(scores.sort_values([selection_metric, 'model']).set_index(['target', 'model']))
    print(f'{target} validation selection metric: {selection_metric}')
print('Validation-selected models:', comparison_report['selected_models'])
for name, result in comparison_report['results'].items():
    print(f'{name} training metadata:', result.get('training_metadata', {}))

fig, axes = plt.subplots(1, len(target_columns), figsize=(5 * len(target_columns), 4), squeeze=False)
for index, target in enumerate(target_columns):
    scores = validation_results[validation_results.target == target].sort_values(['rmse', 'model'])
    axes[0, index].barh(scores['model'], scores['rmse'])
    axes[0, index].invert_yaxis()
    axes[0, index].set_title(f'{target}: validation RMSE')
    axes[0, index].set_xlabel('RMSE (physical units)')
fig.tight_layout()
fig.savefig(RUN_OUTPUT / 'validation_rmse.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig)

## Frozen-winner test report

Only the validation winners are evaluated here; selection is already frozen in `run.json`
and test scores never rerank the queue. Headline median relative error with absolute
RMSE and R-squared alongside; mean relative error stays a secondary table metric.

In [ ]:
test_results = evaluate_selected_models(
    RUN_OUTPUT, split.test[feature_columns], split.test[target_columns],
    target_names=target_columns,
)
for row in test_results.itertuples(index=False):
    print(f'{row.target}: test median relative error {row.median_relative_error:.2%}; '
          f'RMSE {row.rmse:.6g}; R-squared {row.r2:.4f} ({row.model})')
display(test_results.set_index(['target', 'model']))

## Honest evaluation

A single R-squared per target hides what a surrogate user needs: how error splits pre- and
post-stall, how it varies across the eight Reynolds numbers, which airfoils fail worst,
and what all of it does to `L/D`, which amplifies drag error. Train-domain bounds,
outside-train geometry statistics, easy-subset/Re-matched metrics and error
concentration are saved with the full report, without filtering the frozen test.

In [ ]:
from src.training.analysis import lift_to_drag, prediction_frame, stall_regime, summarize

selected_models = comparison_report['selected_models']
predictions = np.empty((len(split.test), len(target_columns)))
for model_name in dict.fromkeys(selected_models.values()):
    scorer = PhysicalPredictor(
        joblib.load(RUN_OUTPUT / model_name / f'{model_name}.joblib'),
        target_columns, TARGET_TRANSFORMS,
    )
    predicted = np.asarray(scorer.predict(split.test[feature_columns])).reshape(
        len(split.test), len(target_columns),
    )
    for index, target in enumerate(target_columns):
        if selected_models[target] == model_name:
            predictions[:, index] = predicted[:, index]
    del scorer, predicted
    gc.collect()

scored = prediction_frame(split.test, predictions, target_columns)
report = summarize(scored, target_columns, training_frame=split.train)
(RUN_OUTPUT / 'analysis.json').write_text(json.dumps(report, indent=2, allow_nan=False) + '\n')

for target in target_columns:
    print(f'--- {target} ---')
    display(pd.DataFrame(report['targets'][target]['regime']).set_index('regime'))
print('L/D over the test split:', report['lift_to_drag'])
display(pd.DataFrame(report['targets']['Cd']['worst_geometries']).set_index('geometry_id'))

In [ ]:
regime = stall_regime(scored)
sample = scored.sample(n=min(5000, len(scored)), random_state=42)
fig, axes = plt.subplots(2, len(target_columns), figsize=(5 * len(target_columns), 9), squeeze=False)
for index, target in enumerate(target_columns):
    upper = axes[0, index]
    for label, marker in (('pre-stall', 'o'), ('post-stall', 'x')):
        rows = sample[regime.loc[sample.index] == label]
        upper.scatter(
            rows[target], rows[f'{target}_predicted'],
            s=8, alpha=.35, marker=marker, label=label,
        )
    limits = [
        min(sample[target].min(), sample[f'{target}_predicted'].min()),
        max(sample[target].max(), sample[f'{target}_predicted'].max()),
    ]
    upper.plot(limits, limits, '--', color='black', linewidth=1)
    upper.set_title(f'{target}: {selected_models[target]}')
    upper.set_xlabel(f'Actual {target}')
    upper.set_ylabel(f'Predicted {target}')
    upper.legend(loc='upper left', fontsize=8)
    upper.grid(alpha=.3)

    lower = axes[1, index]
    by_reynolds = pd.DataFrame(report['targets'][target]['reynolds'])
    lower.bar(by_reynolds['Re'].astype(str), by_reynolds['mae'])
    lower.set_title(f'{target}: MAE by Reynolds number')
    lower.set_xlabel('Re')
    lower.set_ylabel('MAE (physical units)')
    lower.tick_params(axis='x', rotation=45)
fig.suptitle(f'{len(sample):,} sampled test rows; metrics use all {len(scored):,} rows')
fig.tight_layout()
fig.savefig(RUN_OUTPUT / 'regime_and_reynolds.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig)

ratios = lift_to_drag(scored)
plot_rows = ratios.sample(n=min(5000, len(ratios)), random_state=42)
fig, axis = plt.subplots(figsize=(6, 5))
axis.scatter(plot_rows['lift_to_drag'], plot_rows['lift_to_drag_predicted'], s=8, alpha=.35)
limits = [plot_rows['lift_to_drag'].min(), plot_rows['lift_to_drag'].max()]
axis.plot(limits, limits, '--', color='black', linewidth=1)
axis.set_title('Lift-to-drag ratio, selected models')
axis.set_xlabel('Actual L/D')
axis.set_ylabel('Predicted L/D')
axis.grid(alpha=.3)
fig.tight_layout()
fig.savefig(RUN_OUTPUT / 'lift_to_drag.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig)
print(f'All v2 results saved under: {RUN_OUTPUT}')
for path in sorted(RUN_OUTPUT.rglob('*')):
    if path.is_file():
        print(f'  {path.relative_to(RUN_OUTPUT)}  {path.stat().st_size:,} bytes')